# Interactive play (mode 1)

Have a live conversation with the agent over **one persistent game**. The
**model dropdown** at the top of the panel switches between the registry
models (Gemma 4 E4B by default; see `MODEL_CANDIDATES.md`) — switching is an
explicit **Switch model** press, and checking **"Save only one set of weights
at a time"** makes a switch restart the conversation and delete every other
registry model's cached weights from disk.

Asking the agent to play starts a **multi-move turn**: making a move does *not*
end its turn. The agent emits a move token (`[CLOCK]`, `[ANTICLOCK]`, `[FORWARD]`);
generation stops the instant that token appears, the board is re-rendered and fed
back, and it keeps going (e.g. `[CLOCK] [CLOCK] [FORWARD] ...`) until it collects
the gold, ends its turn (finishes a reply without a move token — Gemma's native
end-of-turn), or hits the step cap. You watch every intermediate frame and move
as it happens. The **Restart conversation** button re-initializes the env (a
brand new bare level) and starts a fresh conversation thread.

Prereqs:
- Neo4j is up (`bash scripts/vast_neo4j_launch.sh`, or `docker compose up -d neo4j`).
- The semantic model is seeded once (`python -m agent.runner seed`).
- `bash scripts/setup_env.sh` (installs deps incl. `ipywidgets`, and downloads
  the spaCy + GLiNER weights NAMS' extraction pipeline needs — plain
  `pip install -r requirements.txt` is not enough on its own).

The first cell connects NAMS and loads Gemma, so it can take a minute.

In [1]:
import os
import sys

# Run from the repo root so `agent` imports and `.env` resolve.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from agent.interactive import InteractiveSession

# Connects NAMS on a background loop and loads Gemma 4 E4B (first run is slow).
# Logging is on by default: every LLM generate call and every DB retrieval for
# this session is written under session.logger.run_dir (see the "Dump DB
# status" button in the panel below).
session = InteractiveSession()
print("session_id:", session.session_id)
if session.logger is not None:
    print("log dir:", session.logger.run_dir)

pygame 2.6.1 (SDL 2.28.4, Python 3.12.13)
Hello from the pygame community. https://www.pygame.org/contribute.html


/venv/main/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

session_id: 13aa47f8dd534a5dac3ee76434dadb23
log dir: logs/play_2026-07-24_18-03-25


In [2]:
import ipywidgets as widgets
from IPython.display import Image, clear_output, display

from agent.notebook_ui import model_picker, tame_shift_enter

question_box = widgets.Textarea(
    value="",
    placeholder="Ask the agent something, or tell it to make a move...",
    description="You:",
    layout=widgets.Layout(width="600px", height="70px"),
)
ask_btn = widgets.Button(description="Ask", button_style="primary")
restart_btn = widgets.Button(description="Restart conversation", button_style="warning")
dump_btn = widgets.Button(description="Dump DB status", button_style="info")
out = widgets.Output()

# Frames are rendered at cfg.game_size (768x768 by default) but always shown
# at this fixed width so the conversation flow stays compact and scannable.
FRAME_WIDTH = 420  # px


def _show_frame(path, caption):
    if path:
        print(caption)
        display(Image(filename=path, width=FRAME_WIDTH))


def _show_step(step):
    """Render one step of a multi-move turn as it happens (wired to
    ``session.ask(on_step=...)``)."""
    if step.get("kind") == "reflection":
        print(f"\n--- REFLECTION (after {step['step']} moves) ---")
        print(step["reflection"])
        print("--- end reflection ---\n")
        return
    i = step["step"]
    if i == 0:
        _show_frame(step["before_path"], "-- frame the agent saw --")
    for s in step.get("searches", []):
        print(f"  [SEARCH {s['query']}]")
    print(f"Agent: {step['raw']}")
    if step["action"]:
        print(
            f"[move {i}: {step['action']}  "
            f"gold_collected={step['gold_collected']}  "
            f"gold_remaining={step['gold_remaining']}]"
        )
        # The 'after' frame of this move is the 'before' frame of the next one,
        # so showing afters gives a clean filmstrip of the whole turn.
        _show_frame(step["after_path"], f"-- board after move {i} ({step['action']}) --")
    elif step.get("bare_move"):
        print(
            f"[FORMAT ERROR: bare '{step['bare_move']}' without brackets -- "
            f"NOT a move; turn ends]"
        )
    else:
        print("[no move: answer / end of turn]")
    if step.get("missing_think_close"):
        print(
            "[FORMAT ERROR: thinking model never closed its think block -- "
            "full raw text kept visible]"
        )


def on_ask(_):
    q = question_box.value.strip()
    if not q:
        return
    ask_btn.disabled = True
    try:
        with out:
            print(f"\n=== You: {q}")
            result = session.ask(q, on_step=_show_step)
            if result["solved"]:
                print(f"\n*** Gold collected in {result['num_steps']} move(s)! ***")
            else:
                print(
                    f"\n[turn ended after {result['num_steps']} step(s); "
                    f"gold_remaining={result['gold_remaining']}]"
                )
        question_box.value = ""
    finally:
        ask_btn.disabled = False


def on_restart(_):
    info = session.restart()
    with out:
        clear_output()
        print(f"New conversation. session_id={info['session_id']}")
        _show_frame(info["frame_path"], "-- fresh board --")


def on_dump(_):
    """Snapshot the whole memory graph to a .dump JSON file in this run's log
    directory. Reads over the live connection -- safe at any point, does NOT
    stop Neo4j. Embeddings are dropped (huge, not human-useful). Logical
    snapshot for inspection, distinct from the native binary `neo4j-admin`
    dump produced by `scripts/neo4j_db.sh save`."""
    dump_btn.disabled = True
    try:
        with out:
            info = session.dump_db()
            print(f"\nDB dumped -> {info['path']}")
            print(f"  nodes = {info['nodes']}   relationships = {info['relationships']}")
            if session.logger is not None:
                print(f"  llm log: {session.logger.llm_txt}")
                print(f"  db  log: {session.logger.db_txt}")
    finally:
        dump_btn.disabled = False


ask_btn.on_click(on_ask)
restart_btn.on_click(on_restart)
dump_btn.on_click(on_dump)


def _on_model_switched(info):
    """Refresh the panel after a model switch (the picker already printed the
    switch details into its own status area)."""
    with out:
        if info["restarted"]:
            clear_output()
            print(f"New conversation under {info['label']}. "
                  f"session_id={session.session_id}")
            _show_frame(session.current_frame_path(), "-- fresh board --")
        else:
            print(f"\n[model switched to {info['label']} -- conversation continues]")


# Model picker on top, then conversation, then input: the box stays next to
# the newest output, no scrolling back up after a long exchange.
display(widgets.VBox([
    model_picker(session, on_switched=_on_model_switched),
    out,
    question_box,
    widgets.HBox([ask_btn, restart_btn, dump_btn]),
]))
# Shift+Enter = plain newline in the box (never submits, never re-runs the cell).
tame_shift_enter(question_box)

# Show the current starting board.
with out:
    print(f"session_id={session.session_id}")
    _show_frame(session.current_frame_path(), "-- starting board --")

### Restore the DB to "semantic seeding only" (optional cleanup)

A conversation writes episodic nodes to Neo4j (`Conversation`, `Message`,
`GameSnapshot`, `ReasoningTrace`/`ReasoningStep`). If a run went badly and you
want the *status quo ante* — a fresh box right after `python -m agent.runner
seed` + `link` — run the cell below. It deletes **all** of those episodic nodes
and keeps only the seeded semantic model (`Entity` + `Preference` nodes and
their relationships), optionally clears the on-disk snapshot images, and starts
a fresh conversation.

The cell is gated by `if False:` so it never runs by accident — flip it to
`if True:` (or run the body manually) when you actually want to reset. Note it
clears **every** conversation, not just the most recent one.

In [ ]:
# Flip to `if True:` to wipe episodic memory and restore the seeded state.
if False:
    import shutil

    # 1. Delete every non-semantic node; keep Entity + Preference (+ their edges).
    deleted = session.reset_memory_to_seed()
    print("deleted nodes:", deleted)

    # 2. (Optional) drop the on-disk snapshot PNGs -- all orphaned after step 1.
    img_dir = session.cfg.image_dir
    if img_dir.exists():
        shutil.rmtree(img_dir, ignore_errors=True)
        print("cleared image dir:", img_dir)

    # 3. Start a fresh conversation thread + board so you can keep playing.
    info = session.restart()
    print("new session_id:", info["session_id"])

When you're done, release the model and close the memory client:

In [ ]:
session.close()